# Knowledge Editing Experiment for Neural Decompilation

This notebook tests whether ROME can improve CodeLlama's handling of Ghidra decompilation artifacts.

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install uv
!uv pip install transformers accelerate sentencepiece

# Clone and install EasyEdit (more reliable than pip install from git)
!git clone https://github.com/zjunlp/EasyEdit.git
%cd EasyEdit
!uv pip install "cython<3" setuptools_scm
!uv pip install --no-build-isolation -r requirements.txt
%cd ..
print("EasyEdit installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 111.4 MB/s eta 0:00:00
Using Python 3.12.12 environment at: /usr
Audited 3 packages in 126ms
Cloning into 'EasyEdit'...
remote: Enumerating objects: 9544, done.
remote: Counting objects: 100% (501/501), done.
remote: Compressing objects: 100% (272/272), done.
remote: Total 9544 (delta 315), reused 270 (delta 229), pack-reused 9043 (from 2)
Receiving objects: 100% (9544/9544), 86.28 MiB | 50.14 MiB/s, done.
Resolving deltas: 100% (5997/5997), done.
/content/EasyEdit
Using Python 3.12.12 environment at: /usr
Resolved 4 packages in 146ms
Prepared 2 packages in 101ms
Uninstalled 1 package in 12ms
Installed 2 packages in 8ms
 - cython==3.0.12
 + cython==0.29.37
 + setuptools-scm==9.2.2
Using Python 3.12.12 environment at: /usr
Resolved 112 packages in 1.60s
Prepared 42 packages in 25.52s
Uninstalled 31 packages in 1.38s
Installed 42 packages in 241ms
 + av==14.2.0
 + blessed==1.25.0
 - datasets==4.0.0
 + datasets==1.18.3
 + fairscale=

In [ ]:
rome_config = """alg_name: "ROME"
model_name: "codellama/CodeLlama-7b-Instruct-hf"
stats_dir: "./data/stats"
device: 0
layers: [5]
fact_token: "subject_last"
v_num_grad_steps: 25
v_lr: 5e-1
v_loss_layer: 31
v_weight_decay: 1e-3
clamp_norm_factor: 4
kl_factor: 0.0625
mom2_adjustment: false
context_template_length_params: [[5, 10], [10, 10]]
rewrite_module_tmp: "model.layers.{}.mlp.down_proj"
layer_module_tmp: "model.layers.{}"
mlp_module_tmp: "model.layers.{}.mlp"
attn_module_tmp: "model.layers.{}.self_attn"
ln_f_module: "model.norm"
lm_head_module: "lm_head"
mom2_dataset: "wikipedia"
mom2_n_samples: 100000
mom2_dtype: "float32"
model_parallel: false
"""

with open("codellama-7b-rome.yaml", "w") as f:
    f.write(rome_config)
print("Created ROME config for CodeLlama-7b-Instruct")

Created ROME config for CodeLlama-7b-Instruct


## 2. Define Experiment

In [ ]:
import sys
sys.path.insert(0, 'EasyEdit')

import gc
import subprocess
import tempfile
import os

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from easyeditor import BaseEditor, ROMEHyperParams

MODEL_ID = "codellama/CodeLlama-7b-Instruct-hf"

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def make_prompt(pseudocode: str) -> str:
    return (
        f"<s>[INST] You are an expert C decompiler.\n"
        f"Refine the following Ghidra pseudocode into valid, compilable C code.\n"
        f"STRICT RESPONSE RULES:\n"
        f"1. Do not write a main function.\n"
        f"2. Keep the exact same function name and arguments.\n"
        f"3. Output ONLY the raw code. Do not use Markdown code blocks (```).\n"
        f"4. Do not output any introductory text or explanations.\n\n"
        f"Pseudocode:\n{pseudocode}\n"
        f"[/INST]\n"
    )

def check_compiles(code: str) -> tuple:
    headers = "#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n"
    typedefs = "typedef unsigned int uint; typedef unsigned long ulong;\n"

    with tempfile.NamedTemporaryFile(mode="w", suffix=".c", delete=False) as f:
        f.write(headers + typedefs + code)
        src_path = f.name

    try:
        result = subprocess.run(
            ["gcc", "-c", "-w", src_path, "-o", "/dev/null"],
            capture_output=True, text=True, timeout=10
        )
        os.remove(src_path)
        return (result.returncode == 0, result.stderr[:100] if result.stderr else "OK")
    except Exception as e:
        if os.path.exists(src_path):
            os.remove(src_path)
        return (False, str(e)[:100])

def generate(model, tokenizer, prompt: str, max_tokens: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "[/INST]" in text:
        return text.split("[/INST]")[-1].strip()
    return text

print("Functions defined!")

/usr/local/lib/python3.12/dist-packages/higher/utils.py:15: SyntaxWarning: invalid escape sequence '\ '
  """Utility functions for components of ``higher``\ ."""
/usr/local/lib/python3.12/dist-packages/higher/optim.py:965: SyntaxWarning: "is" with 'int' literal. Did you mean "=="?
  replacement = v[0] if len(v) is 1 else v[group_idx]
/content/EasyEdit/easyeditor/trainer/algs/higher_utils/utils.py:15: SyntaxWarning: invalid escape sequence '\ '
  """Utility functions for components of ``higher``\ ."""
/usr/local/lib/python3.12/dist-packages/timm/models/hub.py:4: FutureWarning: Importing from timm.models.hub is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/EasyEdit/easyeditor/models/melo/peft_egg/src/peft/tuners/lora.py:233: SyntaxWarning: invalid escape sequence '\.'
  layer_index = re.match(f".*.{pattern}\.(\d+)\.*", key)
/content/EasyEdit/easyeditor/models/melo/peft_egg/src/p

Functions defined!


In [ ]:
EDITS = [
    # simple type mappings
    {
        "prompt_template": "In C code, the Ghidra type {} should be replaced with",
        "subject": "undefined1",
        "target_new": "char",
        "ground_truth": "undefined1",
        "description": "undefined1 -> char",
        "test_pseudocode": "undefined1 func0(undefined1 x) { return x; }",
    },
    {
        "prompt_template": "In C code, the Ghidra type {} should be replaced with",
        "subject": "undefined4",
        "target_new": "int",
        "ground_truth": "undefined4",
        "description": "undefined4 -> int",
        "test_pseudocode": "undefined4 func0(undefined4 x) { return x; }",
    },
    {
        "prompt_template": "In C code, the Ghidra type {} should be replaced with",
        "subject": "undefined8",
        "target_new": "long",
        "ground_truth": "undefined8",
        "description": "undefined8 -> long",
        "test_pseudocode": "undefined8 func0(undefined8 x) { return x; }",
    },
    # context-dependent
    {
        "prompt_template": "In C code, the Ghidra constant {} should be replaced with",
        "subject": "_LC0",
        "target_new": "the actual constant value",
        "ground_truth": "_LC0",
        "description": "_LC0 -> constant (impossible for ROME)",
        "test_pseudocode": "int func0(int x) { return x & _LC0; }",
    },
]

# test cases: mix of simple and complex
TEST_CASES = [
    # simple
    {
        "pseudocode": "undefined1 get_char(undefined1 *ptr) { return *ptr; }",
        "should_not_contain": ["undefined1", "undefined"],
        "description": "undefined1 in pointer context",
    },
    {
        "pseudocode": "undefined4 add(undefined4 a, undefined4 b) { return a + b; }",
        "should_not_contain": ["undefined4", "undefined"],
        "description": "undefined4 arithmetic",
    },
    # complex
    {
        "pseudocode": "int abs_check(float x) { return ((uint)x & _LC0) < 1.0f; }",
        "should_not_contain": ["_LC0"],
        "description": "_LC0 context-dependent (ROME cannot help)",
    },
]

print(f"Defined {len(EDITS)} edits and {len(TEST_CASES)} test cases")

Defined 4 edits and 3 test cases


## 3. Baseline Testing & ROME Editing

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

prompts = [e["prompt_template"] for e in EDITS]
subjects = [e["subject"] for e in EDITS]
targets = [e["target_new"] for e in EDITS]

print("\nEdits to apply:")
for i, e in enumerate(EDITS):
    print(f"  {i+1}. Prompt: '{e['prompt_template']}'")
    print(f"      {e['subject']} -> {e['target_new']}")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]


Edits to apply:
  1. Prompt: 'In C code, the Ghidra type {} should be replaced with'
      undefined1 -> char
  2. Prompt: 'In C code, the Ghidra type {} should be replaced with'
      undefined4 -> int
  3. Prompt: 'In C code, the Ghidra type {} should be replaced with'
      undefined8 -> long
  4. Prompt: 'In C code, the Ghidra constant {} should be replaced with'
      _LC0 -> the actual constant value


In [ ]:
print("Loading ROME hyperparameters...")
hparams = ROMEHyperParams.from_hparams("codellama-7b-rome.yaml")

print("Creating editor (this loads the model)...")
editor = BaseEditor.from_hparams(hparams)

baseline_model = editor.model
baseline_results = {"code_tests": [], "generalization_tests": []}

print("\n--- Baseline: Code Prompts ---")
for edit in EDITS:
    prompt = make_prompt(edit["test_pseudocode"])
    output = generate(baseline_model, tokenizer, prompt)
    artifact_present = edit["subject"].lower() in output.lower()
    compiles, error = check_compiles(output)
    status = "FAIL" if artifact_present else "PASS"
    print(f"[{status}] {edit['description']}")
    print(f"    Artifact present: {artifact_present}, Compiles: {compiles}")
    print(f"    Output: {output[:80]}...")
    baseline_results["code_tests"].append({
        "description": edit["description"],
        "artifact_present": artifact_present,
        "compiles": compiles,
        "output": output[:300]
    })

print("\n--- Baseline: Generalization ---")
for test in TEST_CASES:
    prompt = make_prompt(test["pseudocode"])
    output = generate(baseline_model, tokenizer, prompt)
    has_artifacts = any(s.lower() in output.lower() for s in test["should_not_contain"])
    compiles, error = check_compiles(output)
    status = "FAIL" if has_artifacts else "PASS"
    print(f"[{status}] {test['description']}")
    print(f"    Artifacts present: {has_artifacts}, Compiles: {compiles}")
    print(f"    Output: {output[:80]}...")
    baseline_results["generalization_tests"].append({
        "description": test["description"],
        "has_artifacts": has_artifacts,
        "compiles": compiles,
        "output": output[:300]
    })

# Summary of baseline
baseline_code_clean = sum(1 for t in baseline_results["code_tests"] if not t["artifact_present"])
baseline_code_compile = sum(1 for t in baseline_results["code_tests"] if t["compiles"])
baseline_gen_clean = sum(1 for t in baseline_results["generalization_tests"] if not t["has_artifacts"])
baseline_gen_compile = sum(1 for t in baseline_results["generalization_tests"] if t["compiles"])

print(f"\nBaseline Summary:")
print(f"  Code tests - Artifacts removed: {baseline_code_clean}/{len(EDITS)}, Compiles: {baseline_code_compile}/{len(EDITS)}")
print(f"  Generalization - Artifacts removed: {baseline_gen_clean}/{len(TEST_CASES)}, Compiles: {baseline_gen_compile}/{len(TEST_CASES)}")

Loading ROME hyperparameters...
Creating editor (this loads the model)...
We are creating the logger files


config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


BASELINE: Testing UNEDITED Model

--- Baseline: Code Prompts ---
[PASS] undefined1 -> char
    Artifact present: False, Compiles: True
    Output: int func0(int x) {
    return x;
}...
[PASS] undefined4 -> int
    Artifact present: False, Compiles: False
    Output: void func0(uint32_t x) {
    return x;
}...
[PASS] undefined8 -> long
    Artifact present: False, Compiles: True
    Output: func0(x) {
    return x;
}...
[FAIL] _LC0 -> constant (impossible for ROME)
    Artifact present: True, Compiles: False
    Output: int func0(int x)
{
    return x & _LC0;
}...

--- Baseline: Generalization ---
[PASS] undefined1 in pointer context
    Artifacts present: False, Compiles: True
    Output: void get_char(char *ptr) {
    return *ptr;
}...
[PASS] undefined4 arithmetic
    Artifacts present: False, Compiles: True
    Output: int add(int a, int b) {
return a + b;
}...
[FAIL] _LC0 context-dependent (ROME cannot help)
    Artifacts present: True, Compiles: False
    Output: int abs_check(flo

In [ ]:
print("\n" + "="*60)
print("Applying ROME Edits")
print("="*60)

prompts = [e["prompt_template"].replace("{}", e["subject"]) for e in EDITS]
subjects = [e["subject"] for e in EDITS]
targets = [e["target_new"] for e in EDITS]
ground_truths = [e["ground_truth"] for e in EDITS]

print("\nEdits to apply:")
for i, e in enumerate(EDITS):
    print(f"  {i+1}. '{e['subject']}' -> '{e['target_new']}'")
    print(f"      Prompt: '{prompts[i]}'")

print("\nApplying ROME edits...")
try:
    metrics, edited_model, _ = editor.edit(
        prompts=prompts,
        subject=subjects,
        target_new=targets,
        ground_truth=ground_truths,
        keep_original_weight=True,
    )
    print("\nEdit metrics:")
    for i, m in enumerate(metrics):
        print(f"  {EDITS[i]['description']}: {m}")
    edit_success = True
except Exception as e:
    print(f"\nError during editing: {e}")
    import traceback
    traceback.print_exc()
    edit_success = False
    edited_model = None


Applying ROME Edits

Edits to apply:
  1. 'undefined1' -> 'char'
      Prompt: 'In C code, the Ghidra type undefined1 should be replaced with'
  2. 'undefined4' -> 'int'
      Prompt: 'In C code, the Ghidra type undefined4 should be replaced with'
  3. 'undefined8' -> 'long'
      Prompt: 'In C code, the Ghidra type undefined8 should be replaced with'
  4. '_LC0' -> 'the actual constant value'
      Prompt: 'In C code, the Ghidra constant _LC0 should be replaced with'

Applying ROME edits...


  0%|          | 0/4 [00:00<?, ?it/s]

Executing ROME algorithm for the update: [In C code, the Ghidra type undefined1 should be replaced with] -> [ char]
Cached context templates ['{}', 'The following is a. {}', 'The Coffin. {}', 'Therefore\n    public. {}', 'Therefore 3D. {}', 'Because of the way. {}', 'Because of the way. {}', "I'm a. {}", "I've been. {}", "You'll find. {}", 'You are here:. {}', 'The 1996 U.S. {}', 'The 2017-20. {}', 'Therefore 2016, and the. {}', 'Therefore, the company will be able to make. {}', 'Because we have a great deal of interest in. {}', 'Because I’m not a morning person,. {}', 'I have a feeling that this question might have. {}', "I'm sorry. I don't. {}", 'You are here: Home / Blog /. {}', 'You are here: Home » Features ». {}']
Computing left vector (u)...
Selected u projection object undefined1
Left vector shape: torch.Size([11008])
Computing right vector (v)
Lookup index found: 11 | Sentence: In C code, the Ghidra type undefined1 should be replaced with | Token: 1
Rewrite layer is 5
Tying op

 25%|██▌       | 1/4 [00:13<00:39, 13.14s/it]

loss 0.041 = 0.012 + 0.028 + 0.001 avg prob of [ char] 0.9884950518608093
Delta norm: 22.463001251220703
Change in target norm: 5.615750312805176 to 23.318103790283203 => 17.702354431152344
Division Factor: 3.128370761871338
Right vector norm: 7.180415153503418
Right vector shape: torch.Size([4096])
Deltas successfully computed for ['model.layers.5.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.5.mlp.down_proj.weight']
Executing ROME algorithm for the update: [In C code, the Ghidra type undefined4 should be replaced with] -> [ int]
Computing left vector (u)...
Selected u projection object undefined4
Left vector shape: torch.Size([11008])
Computing right vector (v)
Lookup index found: 11 | Sentence: In C code, the Ghidra type undefined4 should be replaced with | Token: 4
Rewrite layer is 5
Tying optimization objective to 31
Recording initial value of v*
loss 4.923 = 4.923 + 0.0 + 0.0 avg prob of [ int] 0.007614445872604847
loss 3.562 = 3.501 + 0.06 + 0.001 

...

loss 0.029 = 0.004 + 0.024 + 0.001 avg prob of [ int] 0.9957893490791321
Delta norm: 22.701265335083008
Change in target norm: 5.675315856933594 to 23.431106567382812 => 17.75579071044922
Division Factor: 3.3009300231933594
Right vector norm: 6.877233028411865
Right vector shape: torch.Size([4096])
Deltas successfully computed for ['model.layers.5.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.5.mlp.down_proj.weight']
Executing ROME algorithm for the update: [In C code, the Ghidra type undefined8 should be replaced with] -> [ long]
Computing left vector (u)...
Selected u projection object undefined8
Left vector shape: torch.Size([11008])
Computing right vector (v)
Lookup index found: 11 | Sentence: In C code, the Ghidra type undefined8 should be replaced with | Token: 8
Rewrite layer is 5
Tying optimization objective to 31
Recording initial value of v*
loss 6.208 = 6.208 + 0.0 + 0.0 avg prob of [ long] 0.002126900013536215
loss 4.017 = 3.969 + 0.048 + 0.00

 75%|███████▌  | 3/4 [00:31<00:10, 10.04s/it]

loss 0.048 = 0.014 + 0.033 + 0.001 avg prob of [ long] 0.9864926934242249
Delta norm: 23.635786056518555
Change in target norm: 5.908946514129639 to 24.470348358154297 => 18.5614013671875
Division Factor: 3.340773582458496
Right vector norm: 7.074944496154785
Right vector shape: torch.Size([4096])
Deltas successfully computed for ['model.layers.5.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.5.mlp.down_proj.weight']
Executing ROME algorithm for the update: [In C code, the Ghidra constant _LC0 should be replaced with] -> [ the actual constant value]
Computing left vector (u)...
Selected u projection object _LC0
Left vector shape: torch.Size([11008])
Computing right vector (v)
Lookup index found: 12 | Sentence: In C code, the Ghidra constant _LC0 should be replaced with the actual constant | Token: 0
Rewrite layer is 5
Tying optimization objective to 31
Recording initial value of v*
loss 2.895 = 2.895 + 0.0 + 0.0 avg prob of [ the actual constant value] 0.0

100%|██████████| 4/4 [00:41<00:00, 10.47s/it]

loss 0.025 = 0.009 + 0.015 + 0.001 avg prob of [ the actual constant value] 0.9908296465873718
Delta norm: 25.3946590423584
Change in target norm: 6.348665237426758 to 26.33293914794922 => 19.98427391052246
Division Factor: 3.464325428009033
Right vector norm: 7.330333232879639
Right vector shape: torch.Size([4096])
Deltas successfully computed for ['model.layers.5.mlp.down_proj.weight']
New weights successfully inserted into ['model.layers.5.mlp.down_proj.weight']
Metrics Summary:  {'pre': {'rewrite_acc': np.float64(0.0625)}, 'post': {'rewrite_acc': np.float64(1.0)}}

Edit metrics:
  undefined1 -> char: {'pre': {'rewrite_acc': [np.float64(0.0)], 'portability': {}}, 'case_id': 0, 'requested_rewrite': {'prompt': 'In C code, the Ghidra type undefined1 should be replaced with', 'target_new': 'char', 'ground_truth': 'undefined1', 'portability': {}, 'locality': {}, 'subject': 'undefined1'}, 'post': {'rewrite_acc': [np.float64(1.0)], 'locality': {}, 'portability': {}}}
  undefined4 -> int: {

## 4. Test Edited Model

In [ ]:
if edit_success and edited_model is not None:
    results = {"factual_tests": [], "code_tests": [], "generalization_tests": []}

    # Test 1: simple factual prompts
    print("\n--- Test 1: Factual Prompts ---")
    for edit in EDITS:
        prompt = edit["prompt_template"].format(edit["subject"])
        output = generate(edited_model, tokenizer, prompt, max_tokens=20)

        target_found = edit["target_new"].lower() in output.lower()

        status = "PASS" if target_found else "FAIL"
        print(f"[{status}] {edit['description']}")
        print(f"    Prompt: \"{prompt}\"")
        print(f"    Output: \"{output.strip()}\"")
        print(f"    Expected: \"{edit['target_new']}\"")

        results["factual_tests"].append({
            "description": edit["description"],
            "target_found": target_found,
            "output": output[:100]
        })

    # Test 2: full evaluation prompts with edited pseudocode
    print("\n--- Test 2: Evaluation Prompts ---")
    for edit in EDITS:
        prompt = make_prompt(edit["test_pseudocode"])
        output = generate(edited_model, tokenizer, prompt)

        artifact_gone = edit["subject"].lower() not in output.lower()
        compiles, error = check_compiles(output)

        status = "PASS" if artifact_gone and compiles else "FAIL"
        print(f"[{status}] {edit['description']}")
        print(f"    Artifact removed: {artifact_gone}, Compiles: {compiles}")
        print(f"    Output: {output[:80]}...")

        results["code_tests"].append({
            "description": edit["description"],
            "artifact_removed": artifact_gone,
            "compiles": compiles,
            "output": output[:300]
        })

    # Test 3: generalization to different code
    print("\n--- Test 3: Generalization ---")
    for test in TEST_CASES:
        prompt = make_prompt(test["pseudocode"])
        output = generate(edited_model, tokenizer, prompt)

        has_artifacts = any(s.lower() in output.lower() for s in test["should_not_contain"])
        compiles, error = check_compiles(output)

        status = "PASS" if not has_artifacts and compiles else "FAIL"
        print(f"[{status}] {test['description']}")
        print(f"    Artifacts removed: {not has_artifacts}, Compiles: {compiles}")
        print(f"    Output: {output[:80]}...")

        results["generalization_tests"].append({
            "description": test["description"],
            "artifacts_removed": not has_artifacts,
            "compiles": compiles,
            "output": output[:300]
        })
else:
    print("Editing failed, cannot test model.")
    results = None

Testing Edited Model

--- Test 1: Factual Prompts ---
[FAIL] undefined1 -> char
    Prompt: "In C code, the Ghidra type undefined1 should be replaced with"
    Output: "In C code, the Ghidra type undefined1 should be replaced with the type of the variable.

```c
int main(void)
{"
    Expected: "char"
[FAIL] undefined4 -> int
    Prompt: "In C code, the Ghidra type undefined4 should be replaced with"
    Output: "In C code, the Ghidra type undefined4 should be replaced with the correct type.

```c
undefined4 __fastcall sub_4010"
    Expected: "int"
[FAIL] undefined8 -> long
    Prompt: "In C code, the Ghidra type undefined8 should be replaced with"
    Output: "In C code, the Ghidra type undefined8 should be replaced with the appropriate type.

```c
undefined8 *undefined8_1;
undefined8"
    Expected: "long"
[FAIL] _LC0 -> constant (impossible for ROME)
    Prompt: "In C code, the Ghidra constant _LC0 should be replaced with"
    Output: "In C code, the Ghidra constant _LC0 should be rep

## 5. Summary

In [ ]:
if results:
    factual_pass = sum(1 for t in results["factual_tests"] if t["target_found"])
    edited_code_artifact = sum(1 for t in results["code_tests"] if t["artifact_removed"])
    edited_code_compile = sum(1 for t in results["code_tests"] if t["compiles"])
    edited_gen_artifact = sum(1 for t in results["generalization_tests"] if t["artifacts_removed"])
    edited_gen_compile = sum(1 for t in results["generalization_tests"] if t["compiles"])

    print("\n" + "-"*60)
    print("COMPARISON: Baseline vs ROME")
    print("-"*60)
    print(f"{'Metric':<35} {'Baseline':>10} {'ROME':>10} {'Diff':>8}")
    print("-"*60)

    print(f"{'Code - Artifacts Removed':<35} {baseline_code_clean:>10}/{len(EDITS)} {edited_code_artifact:>10}/{len(EDITS)} {edited_code_artifact - baseline_code_clean:>+8}")
    print(f"{'Code - Compiles':<35} {baseline_code_compile:>10}/{len(EDITS)} {edited_code_compile:>10}/{len(EDITS)} {edited_code_compile - baseline_code_compile:>+8}")

    print(f"{'Generalization - Artifacts Removed':<35} {baseline_gen_clean:>10}/{len(TEST_CASES)} {edited_gen_artifact:>10}/{len(TEST_CASES)} {edited_gen_artifact - baseline_gen_clean:>+8}")
    print(f"{'Generalization - Compiles':<35} {baseline_gen_compile:>10}/{len(TEST_CASES)} {edited_gen_compile:>10}/{len(TEST_CASES)} {edited_gen_compile - baseline_gen_compile:>+8}")

    print("-"*60)
    print(f"{'Factual Prompts (ROME only)':<35} {'N/A':>10} {factual_pass:>10}/{len(EDITS)}")
    print("-"*60)


------------------------------------------------------------
COMPARISON: Baseline vs ROME
------------------------------------------------------------
Metric                                Baseline       ROME     Diff
------------------------------------------------------------
Code - Artifacts Removed                     3/4          4/4       +1
Code - Compiles                              2/4          3/4       +1
Generalization - Artifacts Removed           2/3          2/3       +0
Generalization - Compiles                    2/3          2/3       +0
------------------------------------------------------------
Factual Prompts (ROME only)                N/A          0/4
------------------------------------------------------------


In [ ]:
import json

output_data = {
    "model": MODEL_ID,
    "edits": [{"subject": e["subject"], "target": e["target_new"]} for e in EDITS],
    "edit_success": edit_success,
    "metrics": [str(m) for m in metrics] if edit_success else [],
    "baseline_results": {
        "code_tests": baseline_results["code_tests"],
        "generalization_tests": baseline_results["generalization_tests"],
        "summary": {
            "code_artifacts_removed": baseline_code_clean,
            "code_compiles": baseline_code_compile,
            "gen_artifacts_removed": baseline_gen_clean,
            "gen_compiles": baseline_gen_compile,
        }
    },
    "edited_results": results,
    "comparison": {
        "baseline_total_clean": baseline_code_clean + baseline_gen_clean,
        "edited_total_clean": (sum(1 for t in results["code_tests"] if t["artifact_removed"]) +
                               sum(1 for t in results["generalization_tests"] if t["artifacts_removed"])) if results else 0,
        "difference": ((sum(1 for t in results["code_tests"] if t["artifact_removed"]) +
                       sum(1 for t in results["generalization_tests"] if t["artifacts_removed"])) -
                       (baseline_code_clean + baseline_gen_clean)) if results else 0,
    }
}

with open("ke_experiment_results.json", "w") as f:
    json.dump(output_data, f, indent=2)

print("Results saved to ke_experiment_results.json")

# download results
try:
    from google.colab import files
    files.download("ke_experiment_results.json")
except:
    print("(Not in Colab, skipping download)")

Results saved to ke_experiment_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Save Edited Model Weights

Save the ROME-edited model for use in the evaluation pipeline.

In [ ]:
# save the edited model weights for evaluation pipeline
SAVE_DIR = "codellama-7b-rome-edited"

if edit_success and edited_model is not None:
    print(f"Saving edited model to {SAVE_DIR}/...")

    # save the full edited model weights
    edited_model.save_pretrained(SAVE_DIR)
    tokenizer.save_pretrained(SAVE_DIR)

    # save edit metadata alongside the model
    import json
    edit_metadata = {
        "base_model": MODEL_ID,
        "edit_method": "ROME",
        "edits_applied": [{"subject": e["subject"], "target": e["target_new"], "description": e["description"]} for e in EDITS],
        "edit_success": edit_success,
    }
    with open(f"{SAVE_DIR}/edit_metadata.json", "w") as f:
        json.dump(edit_metadata, f, indent=2)

    print(f"Model saved! Contents:")
    !ls -lh {SAVE_DIR}/

    !zip -r {SAVE_DIR}.zip {SAVE_DIR}/
    print(f"\nCreated {SAVE_DIR}.zip for download")

    try:
        from google.colab import files
        files.download(f"{SAVE_DIR}.zip")
    except:
        print("(Not in Colab, skipping download)")
else:
    print("No edited model to save (editing failed)")

Saving edited model to codellama-7b-rome-edited/...
Model saved! Contents:
total 26G
-rw-r--r-- 1 root root  813 Jan 16 14:57 chat_template.jinja
-rw-r--r-- 1 root root  672 Jan 16 14:56 config.json
-rw-r--r-- 1 root root  603 Jan 16 14:57 edit_metadata.json
-rw-r--r-- 1 root root  111 Jan 16 14:56 generation_config.json
-rw-r--r-- 1 root root 4.6G Jan 16 14:56 model-00001-of-00006.safetensors
-rw-r--r-- 1 root root 4.6G Jan 16 14:56 model-00002-of-00006.safetensors
-rw-r--r-- 1 root root 4.6G Jan 16 14:57 model-00003-of-00006.safetensors
-rw-r--r-- 1 root root 4.6G Jan 16 14:57 model-00004-of-00006.safetensors
-rw-r--r-- 1 root root 4.6G Jan 16 14:57 model-00005-of-00006.safetensors
-rw-r--r-- 1 root root 2.6G Jan 16 14:57 model-00006-of-00006.safetensors
-rw-r--r-- 1 root root  24K Jan 16 14:57 model.safetensors.index.json
-rw-r--r-- 1 root root  515 Jan 16 14:57 special_tokens_map.json
-rw-r--r-- 1 root root 1.9K Jan 16 14:57 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Jan 16 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>